# Import libraries and packages

In [ ]:
import numpy as np
import pandas as pd
from pyscf import gto, scf, dft, lo, tools
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
import ipywidgets as widgets
import py3Dmol
import cube_tools
import sys

# Compute wavefunction

In [ ]:
'''
    This cell includes an interactive widgets for directly pasting XYZ coordinates and 
    setting the SCF calculations. Uncomment this cell code to use these functions.
    
'''

'''

from IPython.display import display, clear_output
import sys, traceback
from pyscf import cc

# ---------- UI ELEMENTS ----------

method = widgets.Dropdown(
    options=[("HF (RHF)", "hf"), ("DFT (RKS)", "dft"), ("CCSD (post-HF)", "ccsd")],
    value="dft",
    description="Method:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)

basis = widgets.Text(
    value="cc-pvtz",
    description="Basis:",
    placeholder="e.g. def2-svp, 6-31g*, cc-pvtz",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)

xc = widgets.Text(
    value="b3lyp",
    description="DFT XC:",
    placeholder="e.g. b3lyp, pbe0, wb97x-d",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)

charge = widgets.IntText(
    value=0,
    description="Charge:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="180px"),
)

spin = widgets.IntText(
    value=0,
    description="Spin (2S):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="180px"),
)

use_df = widgets.Checkbox(
    value=True,
    description="Use density fitting",
)

verbose = widgets.IntSlider(
    value=4, min=0, max=9, step=1,
    description="Verbose:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="360px"),
)

run_btn = widgets.Button(description="Run", button_style="success")
stop_btn = widgets.Button(description="Reset output", button_style="")
out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px"})

# Enable/disable XC depending on method
def _update_xc_state(*args):
    xc.disabled = (method.value != "dft")
_update_xc_state()
method.observe(_update_xc_state, names="value")

# ---------- CORE RUNNER ----------
def run_qc(xyz_geom: str):
    """
    Run HF/DFT/CCSD on a molecule defined by an XYZ string.
    Expects xyz_geom like:
        "H 0 0 0\nH 0 0 1\n..."
    """

    with out:
        out.clear_output()
        try:
            # Basic checks
            if not xyz_geom or not xyz_geom.strip():
                raise ValueError("No geometry provided.")

            print("Starting calculation...")
            sys.stdout.flush()

            # Build molecule
            mol = gto.M(
                atom=xyz_geom,
                basis=basis.value.strip(),
                unit="ANG",
                charge=int(charge.value),
                spin=int(spin.value),   # 2S
                verbose=int(verbose.value),
            )
            mol.build()

            print(f"Method: {method.value} | Basis: {basis.value.strip()} | Charge: {charge.value} | Spin: {spin.value}")
            if method.value == "dft":
                print(f"XC: {xc.value.strip()}")
            print("Building SCF object...")
            sys.stdout.flush()

            # Run method
            if method.value == "hf":
                mf = scf.RHF(mol)
                mf.verbose = int(verbose.value)
                if use_df.value:
                    mf = mf.density_fit()
                e = mf.kernel()

            elif method.value == "dft":
                mf = dft.RKS(mol)
                mf.xc = xc.value.strip()
                mf.verbose = int(verbose.value)
                if use_df.value:
                    mf = mf.density_fit()
                e = mf.kernel()

            elif method.value == "ccsd":
                # CCSD requires an HF reference first
                mf = scf.RHF(mol)
                mf.verbose = int(verbose.value)
                if use_df.value:
                    mf = mf.density_fit()
                mf.kernel()

                mycc = cc.CCSD(mf)
                mycc.verbose = int(verbose.value)
                ecc, t1, t2 = mycc.kernel()
                e = mf.e_tot + ecc

            else:
                raise ValueError(f"Unknown method: {method.value}")

            print("\n Finished")
            print(f"Total energy (Eh): {e:.12f}")
            sys.stdout.flush()

            return mol, mf

        except Exception:
            print("\n Error:")
            traceback.print_exc()
            sys.stdout.flush()
            return None, None

# ---------- GEOMETRY INPUT ----------
# Geometry entry widget.
geom = widgets.Textarea(
    value="H 0.0 0.0 0.0\nH 0.0 0.0 0.7408",
    description="Geometry:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="780px", height="160px"),
)

# ---------- RENDER MOLECULE ----------
out_render = widgets.Output()
# --- function to render molecule ---
def render_molecule(_=None):
    with out_render:
        out_render.clear_output()
        try:
            xyz = geom.value        
            if not xyz:
                print("No geometry provided.")
                return
            
            # convert coordinates line to XYZ block
            lines = [l.strip() for l in xyz.strip().splitlines() if l.strip()]
            n = len(lines)
            xyz_block = f"{n}\nmolecule\n" + "\n".join(lines)

            global molblock, mol_xyz
            # RDKit: XYZ -> Mol
            mol_xyz = Chem.MolFromXYZBlock(xyz_block)

            # Ensure the molecule has a conformer
            if mol_xyz.GetNumConformers() == 0:
                print("No conformer found in RDKit molecule (cannot render coordinates).")
                return

            # IMPORTANT: XYZ has no bonds; infer connectivity
            # This works well for most organic systems.
            Chem.rdDetermineBonds.DetermineConnectivity(mol_xyz)

            # Now MolBlock should include bonds
            molblock = Chem.MolToMolBlock(mol_xyz)

            view = py3Dmol.view(data=molblock, width=400, height=300)
            view.setStyle({"stick": {}, "sphere": {"scale": 0.3}})

            # Add atom index labels
            conf = mol_xyz.GetConformer()
            for atom in mol_xyz.GetAtoms():
                idx = atom.GetIdx()
                pos = conf.GetAtomPosition(idx)
                view.addLabel(str(idx), {
                    "position": {"x": pos.x, "y": pos.y, "z": pos.z},
                    "fontColor": "black",
                    "fontSize": 14,
                    "showBackground": False,
                })

            view.zoomTo()
            view.show()

        except Exception:
            traceback.print_exc()

render_button = widgets.Button(description="Render", button_style="info")
render_button.on_click(render_molecule)

# ---------- BUTTON HANDLERS ----------
# Make it re-run-safe: clear old handlers then attach once.
run_btn._click_handlers.callbacks.clear()
stop_btn._click_handlers.callbacks.clear()

def on_run(_):
    # Run and store results in globals for later notebook cells (optional)
    global mol_quantum, mf
    mol_quantum, mf = run_qc(geom.value)

def on_reset(_):
    with out:
        out.clear_output()

run_btn.on_click(on_run)
stop_btn.on_click(on_reset)

# ---------- LAYOUT ----------
controls_row1 = widgets.HBox([method, charge, spin])
controls_row2 = widgets.HBox([basis, xc])
controls_row3 = widgets.HBox([use_df, verbose, run_btn, stop_btn])

ui = widgets.VBox([
    geom,
    render_button,
    out_render,
    controls_row1,
    controls_row2,
    controls_row3,
    out
])

display(ui)
'''

In [ ]:
# ---------- READ COORDINATES ---------- #
filepath = "examples/guanine.txt"
with open(filepath, "r") as f:
    lines = [l.strip() for l in f if l.strip()]
xyz = "\n".join(lines)
# convert coordinates line to XYZ block
n = len(lines)
xyz_block = f"{n}\nmolecule\n" + "\n".join(lines)
mol_xyz = Chem.MolFromXYZBlock(xyz_block)
# XYZ has no bonds; determine connectivity.
Chem.rdDetermineBonds.DetermineConnectivity(mol_xyz)
molblock = Chem.MolToMolBlock(mol_xyz)
view = py3Dmol.view(data=molblock, width=400, height=300)
view.setStyle({"stick": {"radius":0.2}, "sphere": {"scale": 0.25}})
# Add atom index labels
conf = mol_xyz.GetConformer()
for atom in mol_xyz.GetAtoms():
    idx = atom.GetIdx()
    pos = conf.GetAtomPosition(idx)
    view.addLabel(str(idx), {
        "position": {"x": pos.x, "y": pos.y, "z": pos.z},
        "fontColor": "black",
        "fontSize": 18,
        "showBackground": False,
    })
view.zoomTo()
view.show()

In [ ]:
# ---------- BUILD MOLECULE ---------- #
mol = gto.M(
           atom=xyz,
           basis="cc-pvdz",
           unit="ANG",
           charge=0,
           spin=0,   # 2S
           )
mol.build()

# ---------- BUILD SCF OBJECT AND RUN CALCULATIONS ---------- #
mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf = mf.PCM()
mf.verbose = 4
mf.kernel()
mf.dump_scf_summary()

In [ ]:
# # ---------- GEOMETRY OPTIMIZATION ---------- #

# from pyscf.geomopt.geometric_solver import optimize
# mol = optimize(mf, maxsteps=100)

# Compute IAO coefficients and do population analysis

In [ ]:
'''
Population Analysis with IAOS
'''

# ---------- BUILD IAOs ---------- #
S_ao = mf.get_ovlp()
# occupied orbitals (works for RHF/RKS)
mo_occ = mf.mo_coeff[:, mf.mo_occ > 0]
C_iao = lo.iao.iao(mol, mo_occ)
C_iao = lo.orth.vec_lowdin(C_iao, S_ao)    # orthonormalize IAOs in AO metric

# Optional: save IAOs to cube files for visualization.
# iao_labels = []
# for i in range(C_iao.shape[1]):
#     tools.cubegen.orbital(mol_quantum, f'iao_{i:02d}.cube'.format(i), C_iao[:,i], nx=80, ny=80, nz=80)
#     iao_labels.append(f'iao_{i:02d}')

In [ ]:
# ---------- BUILD IAOs ---------- #
# transform mo_occ to IAO representation.
mo_occ_iao = C_iao.T @ S_ao @ mo_occ
#constructs the density matrix in the new representation
Dm_iao = mo_occ_iao @ mo_occ_iao.T * 2
iao_mol = mol.copy()
iao_mol.build(False, False, basis='minao')
S_iao = C_iao.T @ S_ao @ C_iao

# ---------- MULLIKEN POPULATION based on IAOs ---------- #
iao_pop, iao_charges = mf.mulliken_pop(iao_mol, Dm_iao, s=S_iao)

In [ ]:
'''
Calculate fragment charges from IAO-based atomic charges
'''

fragA = [ ]  # enter atom indices, e.g., guanine atoms indices in G:C
fragB = [ ]  # enter atom indices, e.g., cytosine atom indices in G:C

Q_A = iao_charges[fragA].sum()
Q_B = iao_charges[fragB].sum()

print(f"Fragment charge A: {Q_A:.5f}")
print(f"Fragment charge B: {Q_B:.5f}")
print(f"Total: {(Q_A + Q_B):.5f}")

In [ ]:
'''
Calculate Mayer bond index from the IAO-based density
'''

# ---------- IAO OFFSET FOR EACH ATOM ---------- #
aoslices = iao_mol.aoslice_by_atom() # return a list with items [start-shell-id, stop-shell-id, start-AO-id, stop-AO-id]
nIAO = iao_mol.nao_nr()
iao2atom = np.zeros(nIAO, dtype=int)
for atom_id, (_, _, ao_start, ao_end) in enumerate(aoslices):
    iao2atom[ao_start:ao_end] = atom_id


# ---------- BOND MATRIX ---------- #
natm = iao_mol.natm
bond_mat = np.zeros((natm, natm))
for A in range(natm):
    ia = np.where(iao2atom == A)[0]
    for B in range(A+1, natm):
        ib = np.where(iao2atom == B)[0]
        # sum_{mu in A, nu in B} P_mu,nu * P_nu,mu
        Pab = Dm_iao[np.ix_(ia, ib)]
        Pba = Dm_iao[np.ix_(ib, ia)]
        bo = np.sum(Pab * Pba.T)
        bond_mat[A, B] = bond_mat[B, A] = float(bo)

# ---------- PRINT BONDS ---------- #
threshold = 0.05  # adjust
data = []
for a in range(natm):
    for b in range(a+1, natm):
        val = bond_mat[a,b]
        if val > threshold:
            data.append({
                "Atom A": f"{mol.atom_symbol(a)}{a}",
                "Atom B": f"{mol.atom_symbol(b)}{b}",
                "Bond index": val
            })
df_bonds = pd.DataFrame(data).sort_values("Bond index", ascending=False)
df_bonds.style.hide()

# Localize and classify IBOs

In [ ]:
'''
Generate IBOs and print them into cube files
'''
# ---------- IBO COEFFICIENTS ---------- #
C_ibo = lo.ibo.ibo(mol, mo_occ, iaos=C_iao)


# ---------- IBOs CUBE FILES ---------- #
ibo_labels = []
for i in range(C_ibo.shape[1]):
    tools.cubegen.orbital(mol, f'ibo_{i:02d}.cube'.format(i), 
                          C_ibo[:,i], nx=80, ny=80, nz=80)
    ibo_labels.append(f'ibo_{i:02d}')

In [ ]:
nocc = C_ibo.shape[1]

# thresholds
lp_max = 0.70
lp_second = 0.20
bond_first = 0.40
bond_second = 0.25

# ---- Compute atom weights on IBOs ----
rows = []
for k in range(nocc):

    # Project IBO into IAO basis
    c_ao = C_ibo[:, k]
    c_iao = C_iao.T @ S_ao @ c_ao

    # Compute weights
    w = np.zeros(natm)
    for a in range(natm):
        idx = np.where(iao2atom == a)[0]
        if idx.size:
            w[a] = float(np.sum(c_iao[idx] ** 2))

    # Get top 3 atoms
    order = np.argsort(w)[::-1]
    top_pairs = [(int(i), float(w[i])) for i in order[:min(3, len(order))]]

    while len(top_pairs) < 3:
        top_pairs.append((None, 0.0))

    (a1, w1), (a2, w2), (a3, w3) = top_pairs

    # Atom labels
    top1 = "" if a1 is None else f"{mol.atom_symbol(a1)}{a1}"
    top2 = "" if a2 is None else f"{mol.atom_symbol(a2)}{a2}"
    top3 = "" if a3 is None else f"{mol.atom_symbol(a3)}{a3}"

    # ---- Classification ----
    if (w1 >= lp_max) and (w2 <= lp_second):
        label = "1_c"
    elif (w1 >= bond_first) and (w2 >= bond_second):
        label = "2_c"
    else:
        label = "multi_c"

    rows.append({
        "IBO": k,
        "Class": label,
        "Top1": top1,
        "w1": w1,
        "Top2": top2,
        "w2": w2,
        "Top3": top3,
        "w3": w3,
    })

df_ibos = pd.DataFrame(rows).sort_values(["Class", "w1"], ascending=[True, False])

df_ibos.round(5).style.hide()

## Plot orbitals

In [ ]:
frontier_labels = ibo_labels
orbital_select = widgets.Dropdown(value=f'{frontier_labels[0]}',options=frontier_labels,description='Orbital: ')

isoval_select = widgets.BoundedFloatText(
    value=0.1,
    min=0,
    max=1.0,
    step=0.01,
    description='Isovalue:',
    style={'description_width': 'initial'}
)

def draw_orbital(label, isoval):
  '''Renders isosurface of selected orbital from calculation.
  Args:
    label (str): Orbital designation, e.g. HOMO, HOMO-1, etc.
  '''

  view = py3Dmol.view(width=500,height=500)
  # save data to dictionary
  grid_data = {}
  with open(f'{label}.cube') as f:
      grid_data[f'{label}'] = f.read()     
      view.addVolumetricData(grid_data[f'{label}'], "cube", {'isoval': isoval, 'color': "lightgreen", 'opacity': 0.8})
      view.addVolumetricData(grid_data[f'{label}'], "cube", {'isoval': -isoval, 'color': "purple", 'opacity': 0.8})

  view.addModel(molblock, 'mol')
  view.setStyle({'stick':{"radius":0.2}, 'sphere': {"scale":0.3}})
  
  # Add atom index labels
  conf = mol_xyz.GetConformer()
  for atom in mol_xyz.GetAtoms():
      idx = atom.GetIdx()
      pos = conf.GetAtomPosition(idx)
      view.addLabel(str(idx), {
          "position": {"x": pos.x, "y": pos.y, "z": pos.z},
          "fontColor": "black",
          "fontSize": 14,
          "showBackground": False,
      })

  view.zoomTo()
  view.show()

widgets.interactive(draw_orbital, {'manual': True, "manual_name": "Render"}, label=orbital_select, isoval=isoval_select)